# 2. Hosp-only preprocessed cohort → hypoglycemia model

This self-contained notebook reads only the handoff from the **hosp-only**
preprocessing notebook. No raw tables are required. The source manifest is
checked, and earlier mixed-source cohorts are rejected.

Candidate inputs are demographics/context, prior lab glucose, other labs,
earlier completed-admission diagnoses and actual concurrent eMAR medications.
No ICU-derived vitals, diet/NPO, weight or feeding-volume predictors are used.

The compact starting architecture retains the earlier regularization settings:
100 trees, ≤4 leaves, minimum leaf size300, L2=50, learning rate0.05. **It must
be reassessed on the new hosp-only cohort**; previous reported metrics are not
applicable. Training-only grouped CV reports its fitting gap. This notebook
does not repeat the old broad hyperparameter search.

Fit preprocessing within each fold, calibrate on separate patients, choose
thresholds on validation patients, then evaluate test patients. Do not tune
on test results. Lab monitoring can miss lows; this predicts documented
hypoglycemia following insulin, not a causal medication effect.

## Setup

In [ ]:
%pip -q install "numpy==2.1.3" "pandas==2.2.3" "scipy==1.15.3" "scikit-learn==1.6.1" "pyarrow==19.0.0" "matplotlib>=3.8,<4" "cloudpickle==3.0.0"

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # Local Jupyter: use local paths in the configuration cell below.

## Paths and analysis settings

Set `DATA_DIR` to notebook 1’s `READY_DIR`. Keep the same patient splits for teammates and subsequent experiments.

In [ ]:
import os
from pathlib import Path

for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "2"
DATA_DIR = Path("/content/drive/MyDrive/BME790/preprocessed_hosp")
RESULTS_DIR = Path("/content/drive/MyDrive/BME790/model_results_hosp")
RUN_GROUP_ABLATION = True
RUN_SENSITIVITIES = True
BOOTSTRAPS = 250  # Resample patients, not individual windows.

## Load the preprocessed data

Load existing split labels without resplitting or changing the cohort. Only a predictor allowlist is used; IDs, labels, monitoring and phenotype variables never enter the model.

In [ ]:
import json
import warnings
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings(
    "ignore", category=UserWarning, module="sklearn.preprocessing._encoders"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
protocol_file = DATA_DIR / "protocol.json"
if not protocol_file.exists():
    raise ValueError(
        "Missing hosp-only protocol.json. Rerun notebook1; old cohorts are not compatible."
    )
protocol = json.loads(protocol_file.read_text())
if (
    protocol.get("source_modules") != ["hosp"]
    or protocol.get("protocol") != "hosp_only_t2dm_v1"
):
    raise ValueError("Expected a hosp-only cohort. Rebuild using the current notebook1.")
frame = pd.read_parquet(DATA_DIR / "primary_cohort.parquet").reset_index(drop=True)
expected = {"train", "validation", "calibration", "test"}
assert set(frame["split"]) == expected, "All four patient splits are required."
assert frame.groupby("subject_id")["split"].nunique().max() == 1, "Patient leakage."
assert set(frame.target.unique()) == {0, 1}
for name in expected:
    assert (
        frame.loc[frame["split"].eq(name), "target"].nunique() == 2
    ), f"{name} needs both outcomes."
display(
    frame.groupby("split").agg(
        patients=("subject_id", "nunique"),
        windows=("target", "size"),
        positive_windows=("target", "sum"),
    )
)
if any(column.startswith("icu_") for column in frame):
    raise ValueError("Unexpected mixed-source columns. Rebuild preprocessing.")
if "source" in frame and not frame.source.eq("emar").all():
    raise ValueError("All index exposures must come from hosp eMAR.")
if frame.groupby("split").target.sum().min() < 10:
    warnings.warn(
        "At least one split has fewer than10 positive windows. "
        "Performance estimates are exploratory and highly unstable."
    )

## Choose strictly pre-exposure inputs

The clinical allowlist covers demographics, glucose, other labs, earlier diagnoses, medications, hosp-derived diagnosis and medication proxies. Sparse/constant inputs are removed using training data only.

In [ ]:
CORE = [
    "age",
    "gender",
    "hours_since_admission",
    "pre_glucose_last",
    "pre_glucose_min",
    "pre_glucose_std",
    "pre_glucose_age_hours",
    "lab_creatinine_last",
    "lab_bun_last",
    "lab_albumin_last",
    "hx_diabetes",
    "hx_kidney_disease",
    "hx_liver_disease",
    "hx_hypoglycemia",
    "med_insulin_records24h",
    "med_steroid_records24h",
    "med_beta_blocker_records24h",
]
EXTRA = [
    "race",
    "admission_type",
    "pre_glucose_count",
    "pre_glucose_slope_per_hour",
    "lab_hba1c_last",
    "lab_sodium_last",
    "lab_potassium_last",
    "lab_bicarbonate_last",
    "lab_lactate_last",
    "lab_hemoglobin_last",
    "lab_white_cells_last",
    "lab_platelets_last",
    "lab_alt_last",
    "lab_ast_last",
    "lab_bilirubin_last",
    "lab_inr_last",
    "med_vasopressor_records24h",
    "med_other_diabetes_drugs_records24h",
    "med_sulfonylurea_records24h",
    "med_dextrose_glucagon_records24h",
    "hx_prior_admissions",
]


def feature_sets(frame):
    full = [
        c
        for c in frame
        if c in ["age", "gender", "race", "admission_type", "hours_since_admission"]
        or c.startswith(("pre_glucose_", "lab_", "med_", "hx_"))
    ]
    return {
        "core": [c for c in CORE if c in full],
        "clinical": [c for c in dict.fromkeys(CORE + EXTRA) if c in full],
        "full": full,
    }


def clean(data, columns):
    data = data.copy()
    for column in columns:
        if pd.api.types.is_numeric_dtype(data[column]):
            data[column] = data[column].replace([np.inf, -np.inf], np.nan)
        else:
            data[column] = data[column].fillna("Missing").astype(str)
    return data


sets = feature_sets(frame)
features = sets["clinical"]
assert len(features) == 38, "Feature contract changed: check notebook1 output version."
frame = clean(frame, sets["full"])
parts = {name: frame.loc[frame["split"].eq(name)].reset_index(drop=True) for name in expected}
train, val, cal, test = [parts[name] for name in ["train", "validation", "calibration", "test"]]

## Preprocessing, classifier and calibration

Clip extreme predictors at fit-set 1st/99th percentiles; impute and scale numeric values; retain useful missing indicators; pool rare categories. Calibration has a positive slope so it cannot reverse risk rankings.

In [ ]:
class StablePreprocessor(BaseEstimator, TransformerMixin):
    """Only fit-set statistics: coverage, clipping, missingness and rare levels."""

    def fit(self, X, y=None):
        self.original_columns_ = list(X.columns)
        self.numeric_ = []
        self.categorical_ = []
        self.dropped_ = {}
        for c in X:
            s = X[c]
            if pd.api.types.is_numeric_dtype(s):
                observed = s.replace([np.inf, -np.inf], np.nan).dropna()
                if len(observed) < 0.10 * len(X) or observed.nunique() < 2:
                    self.dropped_[c] = "<10% observed or constant"
                    continue
                if (
                    set(observed.unique()).issubset({0, 1})
                    and min(observed.eq(0).sum(), observed.eq(1).sum()) < 30
                ):
                    self.dropped_[c] = "binary minority count <30 in fit set"
                    continue
                self.numeric_.append(c)
            elif s.fillna("Missing").astype(str).nunique() > 1:
                self.categorical_.append(c)
        if self.numeric_:
            numeric = X[self.numeric_].replace([np.inf, -np.inf], np.nan).astype(float)
            self.low_ = numeric.quantile(0.01)
            self.high_ = numeric.quantile(0.99)
            clipped = numeric.clip(self.low_, self.high_, axis=1)
            self.medians_ = clipped.median()
            self.scaler_ = StandardScaler().fit(clipped.fillna(self.medians_))
            missing = numeric.isna().mean()
            self.missing_ = [c for c in self.numeric_ if 0.05 <= missing[c] <= 0.95]
        else:
            self.missing_ = []
        if self.categorical_:
            self.encoder_ = OneHotEncoder(
                handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=False
            )
            self.encoder_.fit(X[self.categorical_].fillna("Missing").astype(str))
        self.output_count_ = self.transform(X.iloc[:1]).shape[1]
        return self

    def transform(self, X):
        pieces = []
        if self.numeric_:
            numeric = X[self.numeric_].replace([np.inf, -np.inf], np.nan).astype(float)
            clipped = numeric.clip(self.low_, self.high_, axis=1).fillna(self.medians_)
            pieces.append(self.scaler_.transform(clipped))
            if self.missing_:
                pieces.append(numeric[self.missing_].isna().to_numpy(dtype=float))
        if self.categorical_:
            pieces.append(
                self.encoder_.transform(X[self.categorical_].fillna("Missing").astype(str))
            )
        return np.column_stack(pieces) if pieces else np.zeros((len(X), 1))


class MonotonicCalibration(BaseEstimator):
    """Regularized sigmoid on independent calibration patients; cannot flip ranks."""

    def fit(self, p, y):
        x = logit(p)
        y = np.asarray(y, dtype=float)

        def objective(ab):
            a, b = ab
            z = a + b * x
            value = np.logaddexp(0, z).sum() - np.dot(y, z) + 2 * (b - 1) ** 2
            residual = expit(z) - y
            return value, np.array([residual.sum(), np.dot(residual, x) + 4 * (b - 1)])

        fit = minimize(
            objective, [0, 1], jac=True, bounds=[(-15, 15), (0.05, 5)], method="L-BFGS-B"
        )
        if not fit.success:
            raise ValueError("Calibration optimization did not converge: " + fit.message)
        self.intercept_, self.slope_ = map(float, fit.x)
        return self

    def predict(self, p):
        return expit(self.intercept_ + self.slope_ * logit(p))


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def weights(frame):
    counts = frame.groupby("subject_id").subject_id.transform("size")
    w = 1 / counts.to_numpy(dtype=float)
    return w / w.mean()


def make_model():
    return Pipeline(
        [
            ("preprocess", StablePreprocessor()),
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=100,
                    learning_rate=0.05,
                    max_leaf_nodes=4,
                    max_depth=2,
                    min_samples_leaf=300,
                    l2_regularization=50,
                    max_bins=32,
                    early_stopping=False,
                    random_state=42,
                ),
            ),
        ]
    )

## Check generalization inside training patients

Fixed-architecture, patient-grouped CV. Fit preprocessing from scratch in each fold. Average precision (AP) is the area summarized by the precision–recall ranking; random AP equals event prevalence.

In [ ]:
def rank_metrics(y, p):
    return {"ap": float(average_precision_score(y, p)), "auroc": float(roc_auc_score(y, p))}


folds = list(
    StratifiedGroupKFold(3, shuffle=True, random_state=42).split(
        train, train.target, train.subject_id
    )
)
cv_rows = []
positive_patients = train.loc[train.target.eq(1), "subject_id"].nunique()
if positive_patients < 3:
    raise ValueError("Too few positive training patients for three-fold grouped CV.")
if min(len(tr) for tr, va in folds) < 600:
    warnings.warn(
        "Small hosp-only training folds: min leaf300 may be too restrictive. "
        "Reassess capacity using training-only CV; do not tune on test results."
    )
for fold, (tr, va) in enumerate(folds, 1):
    a, b = train.iloc[tr], train.iloc[va]
    assert set(a.subject_id).isdisjoint(b.subject_id)
    model = make_model().fit(a[features], a.target, model__sample_weight=weights(a))
    for name, data in [("fit", a), ("held_out", b)]:
        cv_rows.append(
            {
                "fold": fold,
                "part": name,
                **rank_metrics(data.target, model.predict_proba(data[features])[:, 1]),
            }
        )
cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(RESULTS_DIR / "grouped_cv.csv", index=False)
display(cv_results.groupby("part")[["ap", "auroc"]].mean())

## Fit, calibrate and freeze the threshold

Fit on training patients, calibrate on calibration patients, and choose a threshold for ≥80% validation sensitivity. Test sensitivity can differ. No test results are used in these choices.

In [ ]:
def threshold(y, p, recall=0.8):
    _, r, t = precision_recall_curve(y, p)
    ok = np.flatnonzero(r[:-1] >= recall)
    return float(t[ok[-1]]) if len(ok) else 0.0


model = make_model().fit(train[features], train.target, model__sample_weight=weights(train))
calibrator = MonotonicCalibration().fit(model.predict_proba(cal[features])[:, 1], cal.target)


def risk(data):
    return calibrator.predict(model.predict_proba(data[features])[:, 1])


cutoff = threshold(val.target, risk(val), recall=0.8)
print(f"Frozen alert threshold: {cutoff:.6f}")

## Evaluate once on the fixed test patients

Report PPV (precision), sensitivity, specificity, alert burden, AUROC, AP and Brier score. Bootstrap whole patients to account for repeated windows. The high-sensitivity threshold may produce many false alerts.

In [ ]:
def operating(y, p, t):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p)
    tn, fp, fn, tp = confusion_matrix(y, p >= t, labels=[0, 1]).ravel()
    r = rank_metrics(y, p) if len(np.unique(y)) == 2 else {"ap": None, "auroc": None}
    return dict(
        n=len(y),
        events=int(y.sum()),
        prevalence=float(y.mean()),
        **r,
        brier=float(brier_score_loss(y, p)),
        threshold=float(t),
        sensitivity=float(tp / (tp + fn)) if tp + fn else None,
        specificity=float(tn / (tn + fp)) if tn + fp else None,
        precision=float(tp / (tp + fp)) if tp + fp else None,
        alert_fraction=float((p >= t).mean()),
        tp=int(tp),
        fp=int(fp),
        fn=int(fn),
        tn=int(tn)
    )


def bootstrap(frame, y, p, t, repeats=BOOTSTRAPS):
    if not len(frame) or len(np.unique(y)) < 2:
        return {}
    groups = list(frame.reset_index(drop=True).groupby("subject_id").indices.values())
    rng = np.random.default_rng(42)
    scores = []
    for _ in range(repeats):
        ix = np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[ix])) < 2:
            continue
        r = operating(y[ix], p[ix], t)
        scores.append(
            [
                r["ap"],
                r["auroc"],
                r["precision"] if r["precision"] is not None else np.nan,
                r["sensitivity"],
            ]
        )
    arr = np.asarray(scores)
    return {
        k: np.nanpercentile(arr[:, j], [2.5, 97.5]).tolist()
        for j, k in enumerate(["ap", "auroc", "precision", "sensitivity"])
    }


reports = []
for name, data in parts.items():
    reports.append(
        {
            "split": name,
            "patients": data.subject_id.nunique(),
            **operating(data.target, risk(data), cutoff),
        }
    )
performance = (
    pd.DataFrame(reports).set_index("split").loc[["train", "validation", "calibration", "test"]]
)
performance.to_csv(RESULTS_DIR / "performance.csv")
display(
    performance[
        [
            "n",
            "events",
            "ap",
            "auroc",
            "precision",
            "sensitivity",
            "specificity",
            "alert_fraction",
            "brier",
        ]
    ]
)
test_ci = bootstrap(test, test.target.to_numpy(), risk(test), cutoff)
(RESULTS_DIR / "test_patient_bootstrap.json").write_text(json.dumps(test_ci, indent=2))
display(pd.DataFrame(test_ci, index=["lower95", "upper95"]).T)
print(
    "Train-test AP gap:",
    round(performance.loc["train", "ap"] - performance.loc["test", "ap"], 4),
)

# Alternative thresholds are also chosen on validation patients, never test.
operating_points = pd.DataFrame(
    [
        {
            "validation_sensitivity_target": recall,
            **operating(test.target, risk(test), threshold(val.target, risk(val), recall)),
        }
        for recall in [0.8, 0.5, 0.2]
    ]
)
operating_points.to_csv(RESULTS_DIR / "operating_points.csv", index=False)
display(
    operating_points[
        ["validation_sensitivity_target", "precision", "sensitivity", "alert_fraction"]
    ]
)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
performance[["ap", "auroc"]].plot.bar(
    ax=ax[0], rot=0, title="Training and held-out performance"
)
precision, recall, _ = precision_recall_curve(test.target, risk(test))
ax[1].plot(recall, precision)
ax[1].axhline(test.target.mean(), linestyle="--", color="gray", label="Event prevalence")
ax[1].set(xlabel="Sensitivity", ylabel="Precision", title="Frozen model: test precision–recall")
ax[1].legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "performance.png", dpi=160)
plt.show()

## Which input groups help?

Exploratory ablation within training patients only: refit the same architecture with all input groups, then omit one group at a time. Positive AP loss means that removal hurts prediction. Correlated groups can substitute for one another; these are predictive associations.

In [ ]:
def group_of(column):
    if column.startswith("pre_glucose_"):
        return "Recent laboratory glucose"
    if column.startswith("lab_"):
        return "Other laboratory tests"
    if column.startswith("hx_"):
        return "Prior diagnoses and admissions"
    if column.startswith("med_"):
        return "Concurrent eMAR medications"
    return "Demographics and admission context"


if RUN_GROUP_ABLATION:
    all_features = sets["full"]
    groups = {
        group: [c for c in all_features if group_of(c) == group]
        for group in sorted(set(map(group_of, all_features)))
    }
    experiments = {
        "All groups": all_features,
        **{
            "Without " + group: [c for c in all_features if c not in columns]
            for group, columns in groups.items()
        },
    }
    ablation = []
    # Different fixed folds for descriptive group analysis; no test tuning.
    group_folds = list(
        StratifiedGroupKFold(3, shuffle=True, random_state=419).split(
            train, train.target, train.subject_id
        )
    )
    for name, columns in experiments.items():
        for fold, (tr, va) in enumerate(group_folds, 1):
            a, b = train.iloc[tr], train.iloc[va]
            fitted = make_model().fit(a[columns], a.target, model__sample_weight=weights(a))
            ablation.append(
                {
                    "configuration": name,
                    "fold": fold,
                    **rank_metrics(b.target, fitted.predict_proba(b[columns])[:, 1]),
                }
            )
        print("Completed:", name)
    ablation = pd.DataFrame(ablation)
    ablation.to_csv(RESULTS_DIR / "group_ablation_folds.csv", index=False)
    group_results = ablation.groupby("configuration")[["ap", "auroc"]].mean()
    group_results["ap_loss_when_removed"] = (
        group_results.loc["All groups", "ap"] - group_results.ap
    )
    group_results = group_results.sort_values("ap_loss_when_removed", ascending=False)
    group_results.to_csv(RESULTS_DIR / "group_ablation.csv")
    display(group_results)

## Check cohort sensitivity without retuning

Use the same frozen primary model, calibrator and threshold. Changing monitoring changes case mix and prevalence, so differences cannot be attributed only to model quality. Small-event results are unstable. Prior-documented T2DM partly addresses retrospective phenotype selection.

In [ ]:
if RUN_SENSITIVITIES:
    sensitivity = []
    for file in sorted(DATA_DIR.glob("*_cohort.parquet")):
        data = clean(pd.read_parquet(file), features)
        data = data.loc[data["split"].eq("test")]
        if len(data):
            sensitivity.append(
                {
                    "cohort": file.stem.removesuffix("_cohort"),
                    "patients": data.subject_id.nunique(),
                    **operating(data.target, risk(data), cutoff),
                }
            )
    sensitivity = pd.DataFrame(sensitivity)
    sensitivity.to_csv(RESULTS_DIR / "cohort_sensitivity.csv", index=False)
    display(
        sensitivity[["cohort", "patients", "events", "ap", "auroc", "precision", "sensitivity"]]
    )

protocol_file = DATA_DIR / "protocol.json"
protocol = json.loads(protocol_file.read_text()) if protocol_file.exists() else {}
# Only call this a new-patient subset if an earlier split manifest was supplied.
if protocol.get("existing_split_manifest_supplied") and "new_patient" in test:
    fresh = test.loc[test.new_patient]
    if len(fresh) and fresh.target.nunique() == 2:
        fresh_result = operating(fresh.target, risk(fresh), cutoff)
        (RESULTS_DIR / "new_patient_test.json").write_text(json.dumps(fresh_result, indent=2))
        display(pd.DataFrame([{"patients": fresh.subject_id.nunique(), **fresh_result}]))

## Save the reusable model

Cloudpickle saves notebook-defined preprocessing/calibration classes with the estimator. Keep this file with its metadata and use matching package versions when loading.

In [ ]:
import sklearn

bundle = {
    "pipeline": model,
    "calibrator": calibrator,
    "features": features,
    "threshold": cutoff,
    "target": "documented glucose <70 mg/dL within next24h after documented insulin",
}
bundle["source_modules"] = ["hosp"]
with (RESULTS_DIR / "model.pkl").open("wb") as file:
    cloudpickle.dump(bundle, file)
metadata = {
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "candidate_features": features,
    "transformed_features": model.named_steps["preprocess"].output_count_,
    "dropped_features": model.named_steps["preprocess"].dropped_,
    "architecture": model.named_steps["model"].get_params(),
    "threshold": cutoff,
}
metadata["source_modules"] = ["hosp"]
(RESULTS_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved results and model:", RESULTS_DIR)